# Step 0 - 파노라마 전처리 (4등분 후 2번째 구간 = 정면)

**목표**: 360° 파노라마(1024×256)를 세로선으로 **4등분**하고, **2번째 구간**을 차량 진행
**Front view**로 사용한다. (가로 중앙은 실제 정면이 아니어서, 2번째 1/4 구간을 정면으로 채택)

**입력**: `test_img/*.jpg` (360° 파노라마)

**출력**: `output/00_front/{stem}_front.jpg` (정면 크롭) + `{stem}_cam.json` (hfov, K, out_hw)

> 2번째 1/4 구간은 360° 중 90° 폭에 해당 → HFOV 90°로 보고 근사 핀홀 K를 산출(03 BEV에서 사용).
> 등각 띠를 그대로 자른 근사이므로 K는 근사값이며, BEV 결과는 상대 지표로 해석한다.

## 1. 라이브러리 및 파라미터

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

# ─── 경로 ──────────────────────────────────
IMG_DIR = Path("test_img")
OUT_DIR = Path("output/00_front")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── 정면 크롭 파라미터 ──────────────────────────
# 파노라마를 가로로 4등분하고 2번째 구간(인덱스 1)을 정면으로 사용.
N_SPLIT = 4          # 가로 분할 수
FRONT_IDX = 1        # 0-based: 2번째 구간

# 2번째 1/4 구간이 차지하는 수평 화각(360° / 4 = 90°)
PANO_HFOV = 360.0
FRONT_HFOV = PANO_HFOV / N_SPLIT   # = 90°

IMG_PATHS = sorted(IMG_DIR.glob("*.jpg"))
print(f"파노라마 {len(IMG_PATHS)}장 발견")
for p in IMG_PATHS:
    print(" ", p.name)

## 2. 4등분 후 2번째 구간 크롭 함수

파노라마 폭을 4등분해 2번째 구간 `[w/4 : w/2]`를 잘라 Front view로 쓴다.

이 구간은 360° 중 90° 폭이므로, 03 BEV의 핀홀 역투영에 쓸 근사 intrinsics를 산출한다.
- `fx = (W/2) / tan(HFOV/2)`, `fy = fx`(정사각 픽셀 가정), `cx=W/2`, `cy=H/2`.

In [2]:
def make_front_crop(pano_bgr, n_split=N_SPLIT, front_idx=FRONT_IDX,
                    front_hfov=FRONT_HFOV):
    """파노라마를 가로 n등분 후 front_idx 구간을 크롭. (크롭 이미지, 근사 K) 반환."""
    ph, pw = pano_bgr.shape[:2]
    seg_w = pw // n_split
    x0 = front_idx * seg_w
    x1 = x0 + seg_w
    front = pano_bgr[:, x0:x1].copy()
    fh, fw = front.shape[:2]

    # 근사 핀홀 intrinsics (구간 폭 = front_hfov)
    fx = (fw / 2.0) / np.tan(np.radians(front_hfov) / 2.0)
    cx, cy = fw / 2.0, fh / 2.0
    K = np.array([[fx, 0, cx], [0, fx, cy], [0, 0, 1]], dtype=np.float64)
    return front, K


print("크롭 함수 정의 완료")

크롭 함수 정의 완료


## 3. 전체 이미지 처리 및 저장

In [ ]:
for img_path in tqdm(IMG_PATHS, desc="전처리"):
    stem = img_path.stem
    pano = cv2.imread(str(img_path))
    ph, pw = pano.shape[:2]

    front, K = make_front_crop(pano)
    fh, fw = front.shape[:2]

    # 저장: 정면 크롭
    out_img = OUT_DIR / f"{stem}_front.jpg"
    cv2.imwrite(str(out_img), front)

    # 저장: 카메라 메타 (02·03이 공유)
    cam = {
        "src_image": img_path.name,
        "src_hw": [ph, pw],
        "out_hw": [fh, fw],
        "hfov_deg": FRONT_HFOV,
        "split": [N_SPLIT, FRONT_IDX],
        "K": K.tolist(),
        "fx": K[0, 0], "fy": K[1, 1], "cx": K[0, 2], "cy": K[1, 2],
    }
    with open(OUT_DIR / f"{stem}_cam.json", "w", encoding="utf-8") as f:
        json.dump(cam, f, ensure_ascii=False, indent=2)

print("\n=== 완료 ===")
print(f"-> next: 01_segmentation.ipynb (input = {OUT_DIR})")